## Hard Fork e Soft Fork
Intorno al 2014-2017 Bitcoin iniziò ad avere un problema serio di scalabilità: arrivavano molte più transazioni, ma ogni blocco aveva un limite massimo di circa 1 MB (questo limite era stato introdotto originariamente da Satoshi per evitare attacchi di spam alla rete). Con l'aumento della domanda, i tempi di conferma diventavano più lunghi e i costi di fee aumentarono.

L'idea Naive per risolvere il problema era semplicemente di aumentare la block size, ma questo avrebbe introdotto un problema importante: blocchi più grandi avrebbero comportato a una blockchain più grande permettendo a meno persone di mantenere un nodo completo, aumentando così il rischio di centralizzazione.

Nel 2017 una parte della comunità decise comunque di aumentare la block size tramite un **Hard Fork**, estendendo la dimensione dei blocchi nella blockchain a 2 MB. Un **Hard Fork modifica le regole del protocollo di consenso in modo NON retrocompatibile, con nuove regole meno restrittive delle vecchie**. 

La peculiarità dell'hard fork è che questo comporta la creazione di una nuova blockchain, dal momento che i nodi che decidono di non aggiornare il software non seguiranno le nuove regole (nel nostro caso specifico, rifiuteranno tutti i blocchi più grandi di 1 MB). In questo modo nacque la catena **Bitcoin Cash**, distaccata dalla chain originale **Bitcoin**.

Quando avviene un hard fork tipicamente una delle due chain è destinata a crollare in borsa, in questo caso Bitcoin Cash che ad oggi sta a circa 300$ contro i 30.000$ di Bitcoin. Un'osservazione importante è tutta la storia precedente al fork è condivisa tra le due chain, quindi chiunque avesse dei fondi non spesi prima del fork si è ritrovato con fondi equivalenti su entrambe le catene. Qui uno dei motivi che portò Bitcoin Cash a crollare: molti degli utenti fedeli a Bitcoin e in disaccordo con il fork venderono tutti i loro Bitcoin Cash in quanto non ne avevano interesse.

Per queste ragioni gli hard fork sono molto rischiosi e difficili da adottare, tanto che nessun hard fork ha mai sostituito la chain originale. Se ci sono aggiornamenti importanti da effettuare al protocollo, è tipicamente necessario realizzare un **Soft Fork**.

Un **Soft Fork** è un aggiornamento che **modifica le regole del protocollo in modo retrocompatibile, con nuove regole più restrittive delle vecchie**. In questo modo i nodi che non aggiornano il software potranno accettare i nuovi blocchi, anche se non ne riconoscono tutte le regole. Inoltre, se c'è consenso sufficiente tra i miner, i vecchi nodi non potranno più produrre blocchi validi.

### **SegWit Segregated Witness** 
SegWit fu il primo grande soft fork di Bitcoin, avvenuto nel 2017. L'upgrade aveva **due obiettivi principali**:
1. **Aumentare la capacità effettiva dei blocchi**, senza farlo esplicitamente come aveva fatto Bitcoin Cash
2. **Risolvere il problema di Transaction Malleability**, un bug che permetteva di modificare l'ID di una transazione prima che fosse confermata


#### **Transaction Malleability**
Partiamo parlando del problema di **Transaction Malleability**. Si ricorda che una transazione Bitcoin, pre SegWit, era composta come segue:

<img src="img/old_tx.jpeg" width="700">

L'identificatore della transazione Bitcoin era calcolato hashando la serializzazione in byte della transazione, formalmente $txid = \text{SHA256(SHA256(tx))}$, per cui modificare anche solo un byte della transazione avrebbe cambiato completamente il suo ID.

Di una vecchia transazione, non era possibile modificare liberamente i dati come input spesi, output creati, amount, destinatari e locktime in quanto ciò avrebbe portato a una transazione non valida, che sarebbe stata scartata dal miner. **L'unico dato che era possibile modificare, rendendo comunque valida la transazione, erano le firme.**

Il problema con le vecchie transazioni è che le firme, presenti negli input della transazione, facevano parte dei dati hashati per costruire il $txid$. Un attaccante quindi poteva modificare le firme cambiando del tutto il $txid$ della transazione ma lasciando invariati tutti gli altri dati, rendendola comunque valida e potenzialmente accettabile da parte di un miner.

(NB. non l'abbiamo visto ma in generale esistono modi per modificare le firme anche da parte di utenti che non possiedono la sk che l'ha generata, per via della malleabilità delle firme ECSDA)

**Questo rappresenta un grosso problema in quanto finché la transazione non entra effettivamente nella blockchain, non posso sapere con certezza quele $txid$ avrà**. Questo rompe del tutto la possibilità di creare applicazioni sicure basate su catene off-chain, come ad esempio la **Lightning Network**.

In breve, la Lightning Network è una rete di canali di pagamento off-chain che permette di effettuare transazioni veloci e a basso costo tra due parti, senza dover attendere la conferma sulla blockchain. Immaginiamo che Mario voglia aprire un canale con Amazon via Lightning Network, Mario ha 100 euro da spendere per Amazon --> Mario 100 --- Amazon 0. Allora viene creata una **funding transaction TX1** sulla blockchain dove Mario blocca 100 euro in output multisig per cui quei 100 euro possono essere spesi solo con la firma di Mario e Amazon. Ma se Amazon sparisse, Mario non potrebbe recuperare i soldi (non ha la firma di amazon) --> prima di pubblicare TX1 Mario si accorda con Amazon creando una transazione di uscita TX2 già firmata per cui Mario riceve 100 e Amazon 0 (**commitment transaction**, non è pubblicata subito, resta off-chain). Poi off chain Mario e Amazon si scambiano soldi, aggiornando di volta in volta la commitment transaction finché non decidono di chiudere il canale, pubblicando l'ultima commitment transaction sulla blockchain. 

Il problema pre-SegWit con la possibilità di creare una Lightning Network affidabile era che se TX1 non era ancora confermata, un attaccante poteva liberamente modificarne la firma, creando una nuova transazione identica ma con $txid$ diverso TX1'. Ma TX2 era già stata firmata con il vecchio $txid$ di TX1, quindi se TX1' veniva inserita nella blockchain per prima, TX2 sarebbe diventata non valida, e Mario avrebbe potenzialmente perso i suoi soldi.

Quindi per risolvere questi problemi i $txid$ **devono essere stabili, non modificabili "a piacere"**.

SegWit risolve il problema con la seguente idea: **separare le firme dalla parte hashata per il calcolo del $txid$**. In particolare, le firme vengono spostate nella nuova sezione **witness** della transazione, che non viene considerata per il calcolo del $txid$. Di seguito la struttura delle nuove transazioni SegWit:

<img src="img/new_tx.jpeg" width="700">

**dove solo la parte senza Marker, Flag e Witness viene hashata per il calcolo del** $txid$. **Marker** e **Flag** sono due byte introdotti da SegWit per questioni di retrocompatibilità: ai nuovi nodi vedere marker 0x00 e flag 0x01 sta a significare che si tratta di una transazione SegWit e che quindi la firma è spostata nella sezione witness.

I vecchi nodi non parseranno correttamente la transazione e ai loro occhi il Marker 0x00 fa apparire la transazione come una una transazione senza input (infatti subito dopo ver nelle vecchie transazioni il primo campo era numero di input, in questo caso marker 0x00).

**Ma come rendere retrocompatibile il witness per i vecchi nodi, facendo in modo che essi accettino la transazione?** Si agisce anzitutto a livello peer to peer: durante l'handshake, un nuovo nodo riconosce se sta comunicando con un nodo pre o post SegWit. Se comunica con un nodo legacy --> gli manda la transazione compatibile legacy, senza witness. Il vecchio nodo come anticipato vedrà marker 0x00 interpretando la tx come senza input, ma come far sì che consideri validi gli output? Gli output SegWit sono costruiti in modo che i vecchi nodi li interpretino come spendibili da chiunque (anyone-can-spend), mentre i nuovi nodi applicano le verifiche crittografiche reali tramite il witness. Così facendo i vecchi nodi considereranno validi anche output SegWit che non saprebbero verificare altrimenti.

Ma in questo modo quindi un vecchio nodo potrebbe spendere qualsiasi output SegWit... **se non fosse che per rendere un soft fork sicuro è necessario che si abbia consenso sufficiente tra i miner!**

In particolare per SegWit si attese che almeno il 95% dei miner segnalasse il supporto per l'aggiornamento, per poi far partire l'upgrade dopo un certo numero di blocchi. Perché 95%? Di modo che anche se ci fossero stati alcuni miner in disaccordo, questi non sarebbero riusciti a stare al passo nella creazione dei nuovi blocchi a causa della Proof of Work e sarebbero "rimasti indietro" nella blockchain. SegWit divenne ufficialmente attivo nell'agosto del 2017, con il consenso del 100% dei miner.

Quindi, anche se un nodo vecchio tentasse di spendere un qualsiasi output SegWit, i miner non accetterebbero la transazione, in quanto non rispetta le nuove regole di consenso. 


**N.B** - il punto di SegWit è che è un soft fork --> retrocompatibile, non ha eliminato il vecchio formato, ha "solo" introdotto un nuovo tipo di transazione spendibile tramite witness. Quindi le transazioni legacy continuano ad esistere e se sono coerenti con le regole di consenso continuano ad essere accettate dai miner!

#### **Aumento della capacità dei blocchi**
L'intuizione qui è che sono le firme ad occupare la maggior parte dello spazio in una transazione Bitcoin (ogni firma sono 64 byte). Le firme sono necessarie al momento della validazione, **ma dopo che la transazione è stata verificata e inserita nella blockchain sono meno importanti degli altri dati storici**. 

L'idea di SegWit è quindi di "far pesare meno" la sezione in cui le firme sono presenti, il witness. Il vecchio limite di 1MB per blocco viene sostituito con una nuova misura: **4 milioni di weight units**, dove la weight unit è calcolata secondo le seguenti regole:
- la parte non witness della transazione pesa $4 \cdot \text{byte non witness}$
- la parte witness della transazione pesa $1 \cdot \text{byte witness}$

$$\text{weight} = 4 \cdot \text{byte non witness} + 1 \cdot \text{byte witness}$$
**La conseguenza pratica di questo ragionamento è che dato che le firme, che occupavano tantissimo, ora costano 4 volte di meno, ora è possibile inserire più transazioni nello stesso blocco**. 

In questo modo un blocco può fisicamente superare il limite di 1MB, in particolare un blocco SegWit arriva a circa 4 MB, ma solo presenta quasi solo transazioni SegWit, quindi tantissimi witness. Nella pratica, i blocchi ad oggi in media sono circa 1.3-2 MB circa.

Questo resta un soft fork perché come spiegato prima i vecchi nodi vedono solo la parte legacy delle transazioni anche nei blocchi, che restano quindi comunque di dimensione inferiore a 1MB, quindi ai loro occhi il limite non è stato violato.

### Ricapitolando
**Domanda**: se i miner si sono organizzati al 95% per accettare il soft fork, non si poteva fare lo stesso con un hard fork?

Sì, ma c'è una differenza fondamentale:
- con un Soft Fork i vecchi nodi continuano comunque a seguire la nuova chain, anche se non ne comprendono tutte le nuove regole. Quindi la rete resta una sola, i nuovi nodi non vengono automaticamente esclusi. (i nodi vecchi vedono i blocchi in modo diverso, legacy, ma è la stessa catena!)
- con un Hard Fork invece i vecchi nodi considerano invalidi i nuovi blocchi, per cui anche se una piccola minoranza continua con il vecchio software, nasce una chain separata, ed una delle due è destinata a crollare in borsa

Usare un soft fork significa minimizzare il rischio di split permanente e non obbliga tutti i nodi ad aggiornare subito il software.